# Milestone 3 — Model-Guided Optimization of a CUDA GEMM on a Turing GPU

**Project:** Optimizing a Custom CUDA GEMM Kernel Toward cuBLAS-Level Performance
**Files needed in the Colab file explorer:** `gemm_cuda_opt.cu` (and, optionally, `results_milestone2.csv` from Milestone 2 for the comparison table).
**Runtime:** GPU → T4. Expect the full run to take roughly 10–20 minutes.

### What changed compared with Milestones 1–2
Milestone 2 *hypothesised* that the naive kernel was limited by off-chip bandwidth. Milestone 3 turns the project from "apply known optimizations and watch the numbers go up" into **testing a quantitative model against measurements**:

| RQ | Question | Notebook section |
|---|---|---|
| RQ1 | How well does a per-level traffic model (FMA / shared memory / DRAM) predict each optimization stage on a T4, and was our Milestone-2 explanation right? | §3 |
| RQ2 | How sensitive is the register-blocked kernel to tile configuration, and can the model prune the search space? | §4 |
| RQ3 | Where does the Tensor-Core error come from (input quantization vs. accumulation), and how does it scale with K? | §5 |
| RQ4 | How much host↔device transfer can be hidden by panel pipelining, and does measured overlap match the $\max(\cdot)$ model? | §6 |
| RQ5 | Is there a structural reason a custom kernel can still beat cuBLAS (fused epilogue)? | §7 |

**Honesty notes.** (i) All numbers in the report come from *this* notebook's output — nothing is pre-filled. (ii) Only one GPU is available, so the "distributed" part is a **single-GPU, stream-based proxy** (panel decomposition + copy/compute overlap), not a multi-GPU measurement. (iii) No Nsight counters here; per-level attributions are *model-based* and are validated with hardware counters in Milestone 4.

## 0. Environment and build

In [ ]:
import subprocess, os
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
print(sh("nvidia-smi --query-gpu=name,compute_cap,clocks.max.sm,clocks.max.mem,memory.total --format=csv"))
cc = sh("nvidia-smi --query-gpu=compute_cap --format=csv,noheader").splitlines()[0].replace(".", "")
ARCH = "sm_" + cc
print("Using", ARCH)
assert os.path.exists("gemm_cuda_opt.cu"), "Upload gemm_cuda_opt.cu to the Colab file explorer first."

In [ ]:
!nvcc -O3 -arch=$ARCH -std=c++14 -lineinfo -o gemm_cuda_opt gemm_cuda_opt.cu -lcublas

## 1. Measure the machine (device ceilings)
DRAM copy bandwidth, achievable FP32 FMA peak, and the **SM clock actually observed** during the FMA run. The T4 is a 70 W part and Colab GPUs are shared, so the clock (and therefore every ceiling) can sit well below the datasheet — the model must use these measured values.

In [ ]:
!./gemm_cuda_opt micro

## 2. Correctness validation
Same Frobenius-norm methodology and thresholds as Milestones 1–2, now over **11 shapes (3 rectangular, several not multiples of any tile size)** and **every kernel**: naive, uncoalesced control, smem tiling, 16 register-blocked configurations, and the WMMA kernel. The FP16 path gets its own, justified thresholds plus an *accumulation-only* check against a reference computed on FP16-rounded inputs.

In [ ]:
!./gemm_cuda_opt validate

## 3. Optimization ladder vs. cuBLAS (RQ1)
Naive → smem tiling → 2-D register blocking → float4 loads → Tensor Cores, benchmarked against cuBLAS SGEMM and cuBLAS FP16-Tensor-Op GEMM. Timing = 1 untimed warm-up + adaptive repetitions, **median** reported (Milestone 2 used a single cold launch).

In [ ]:
!./gemm_cuda_opt bench

## 4. Tile-configuration sweep (RQ2)
16 configurations (8 tile shapes × scalar/float4 loads). Registers, local-memory spills and resident blocks/SM are queried from the runtime.

In [ ]:
!./gemm_cuda_opt tune

## 5. FP16 Tensor-Core error budget (RQ3)

In [ ]:
!./gemm_cuda_opt precision

## 6. Panel-pipelined streaming GEMM — single-GPU proxy for distributed overlap (RQ4)
B stays resident on the device; C is split into row panels, each panel doing H2D(A panel) → kernel → D2H(C panel) on one of S streams. Compared with the Milestone-2-style serial, pageable-memory flow. **One GPU only** — this models the compute/communication overlap idea of Uncut-GEMMs, not multi-GPU scaling. Needs ~2 GB of pinned host RAM at N=8192.

In [ ]:
!./gemm_cuda_opt pipe

## 7. Fused GEMM + bias + ReLU epilogue (RQ5)

In [ ]:
!./gemm_cuda_opt fuse

## 8. Analysis, figures, LaTeX tables and macros
Everything below reads the CSVs written above and emits: `fig_*.pdf`, `tab_*.tex`, and `results_macros.tex` (numbers used in the report's prose). Upload them next to `Milestone_3.tex` and compile — the report contains no hand-typed results.

*Optional:* upload `results_milestone2.csv` before running §8 to fill the Milestone-2 vs. Milestone-3 comparison table.

In [ ]:
# ============================================================
# ANALYSIS 0 -- setup, measured device ceilings
# ============================================================
import os, glob, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

plt.rcParams.update({"font.size": 8, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.axisbelow": True, "savefig.bbox": "tight"})
MAC = {}          # LaTeX macro name -> string, written to results_macros.tex

def setm(key, val, fmt="{:.0f}"):
    if isinstance(val, str):
        MAC[key] = val
    elif val is None or (isinstance(val, float) and not np.isfinite(val)):
        MAC[key] = "n/a"
    else:
        MAC[key] = fmt.format(val)

def savefig(name):
    plt.savefig(name + ".pdf"); plt.savefig(name + ".png", dpi=200); plt.show()

def tt(s):                      # LaTeX-safe monospace
    return r"\texttt{" + str(s).replace("_", r"\_") + "}"

micro = pd.read_csv("results_m3_micro.csv").set_index("key")["value"]
P_FMA   = float(micro["measured_fma_gflops"])      # GFLOP/s, achievable FP32 peak
BW_DRAM = float(micro["measured_copy_gbs"])        # GB/s, achievable DRAM bandwidth
BW_SMEM = float(micro["smem_bw_gbs"])              # GB/s, 128 B/clk/SM at observed clock
CLK     = float(micro["observed_clock_ghz"])
NOM_FP  = float(micro["nominal_fp32_gflops"])
print(micro.to_string())

setm("devName", str(micro["device"]))
setm("smCount", float(micro["sm_count"]))
setm("obsClock", CLK, "{:.2f}")
setm("nomFP", NOM_FP / 1000, "{:.2f}")
setm("fmaPeak", P_FMA / 1000, "{:.2f}")
setm("fmaPeakPctNom", 100 * P_FMA / NOM_FP, "{:.0f}")
setm("bwCopy", BW_DRAM, "{:.0f}")
setm("bwNom", float(micro["nominal_dram_gbs"]), "{:.0f}")
setm("smemBW", BW_SMEM / 1000, "{:.2f}")

with open("tab_micro.tex", "w") as f:
    f.write(r"""\begin{tabular}{lrr}
\hline
Quantity & Nominal & Measured \\ \hline
FP32 peak (TFLOP/s) & %.2f & %.2f \\
DRAM bandwidth (GB/s) & %.0f & %.0f \\
SM clock (GHz) & %.2f (max) & %.2f (observed) \\
Shared-mem. BW (TB/s) & --- & %.2f \\ \hline
\end{tabular}""" % (NOM_FP/1000, P_FMA/1000, float(micro["nominal_dram_gbs"]), BW_DRAM,
                    float(micro["max_clock_mhz"])/1000, CLK, BW_SMEM/1000))

In [ ]:
# ============================================================
# ANALYSIS 1 -- ablation ladder + analytical model (RQ1)
# ============================================================
abl = pd.read_csv("results_m3_ablation.csv")
piv = abl.pivot(index="N", columns="kernel", values="gflops")
eps = abl.pivot(index="N", columns="kernel", values="eps_vs_cublas")
NREF = 4096 if 4096 in piv.index else int(piv.index.max())
order = ["naive_uncoalesced", "naive_m2", "smem_tile32", "reg2d_128x128x8_scalar",
         "reg2d_128x128x8_vec", "wmma_fp16_tc", "cublas_sgemm", "cublas_tensorop_fp16"]
short = {"naive_uncoalesced": "naive, uncoalesced", "naive_m2": "naive (M2)",
         "smem_tile32": "smem tile 32", "reg2d_128x128x8_scalar": "reg-block (scalar ld)",
         "reg2d_128x128x8_vec": "reg-block (float4)", "wmma_fp16_tc": "WMMA FP16 TC",
         "cublas_sgemm": "cuBLAS SGEMM", "cublas_tensorop_fp16": "cuBLAS FP16 TC"}
g = lambda k, N=NREF: float(piv.loc[N, k]) if (k in piv.columns and N in piv.index and np.isfinite(piv.loc[N, k])) else float("nan")

fig, ax = plt.subplots(figsize=(3.5, 2.7))
for k in order:
    if k in piv.columns:
        ax.plot(piv.index, piv[k], marker="o", ms=2.5, lw=1, label=short[k])
ax.axhline(P_FMA, ls="--", c="k", lw=0.8)
ax.text(piv.index.min(), P_FMA * 1.08, "measured FP32 FMA peak", fontsize=6)
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("N (square GEMM)"); ax.set_ylabel("GFLOP/s")
ax.legend(fontsize=5.5, ncol=2, loc="lower right")
savefig("fig_ablation")

# ---- analytical model:  P <= min(P_fma, B_smem/beta_smem, B_dram/beta_glob) ----
def b_smem(TM, TN): return 2.0 * (TM + TN) / (TM * TN)       # smem bytes per FLOP
def b_glob(BM, BN): return 2.0 * (1.0 / BM + 1.0 / BN)       # global->SM bytes per FLOP
def ceiling(TM, TN, BM, BN):
    return min(P_FMA, BW_SMEM / b_smem(TM, TN), BW_DRAM / b_glob(BM, BN))

# Milestone-2 hypothesis: "naive kernel is DRAM-bandwidth-bound".
# Every FMA requests two 4-byte operands -> 4 B/FLOP at the load/store level.
BETA_REQ = 4.0
m2_ceiling = BW_DRAM / BETA_REQ                       # GFLOP/s if EVERY request went to DRAM
naive_g = g("naive_m2")
implied_bw = naive_g * BETA_REQ                       # GB/s of operand traffic the kernel actually demanded
setm("m2HypCeil", m2_ceiling, "{:.0f}")
setm("impliedBW", implied_bw / 1000, "{:.2f}")
setm("impliedRatio", implied_bw / BW_DRAM, "{:.1f}")
setm("naiveOverHyp", naive_g / m2_ceiling, "{:.1f}")

rows = []
for k, TM, TN, BM, BN in [("smem_tile32", 1, 1, 32, 32),
                          ("reg2d_128x128x8_scalar", 8, 8, 128, 128),
                          ("reg2d_128x128x8_vec", 8, 8, 128, 128)]:
    c = ceiling(TM, TN, BM, BN)
    rows.append((k, g(k), c))
model_df = pd.DataFrame(rows, columns=["kernel", "measured", "ceiling"])
model_df["eff"] = model_df.measured / model_df.ceiling
print(model_df)

fig, ax = plt.subplots(figsize=(3.5, 2.3))
xs = np.arange(len(model_df)); w = 0.38
ax.bar(xs - w / 2, model_df.ceiling / 1000, w, label="model ceiling")
ax.bar(xs + w / 2, model_df.measured / 1000, w, label="measured")
ax.set_xticks(xs); ax.set_xticklabels([short[k] for k in model_df.kernel], fontsize=6)
ax.set_ylabel("TFLOP/s"); ax.legend(fontsize=6)
savefig("fig_model")
for i, key in enumerate(["smem", "regS", "regV"]):
    setm("eff_" + key, 100 * model_df.eff.iloc[i], "{:.0f}")
    setm("ceil_" + key, model_df.ceiling.iloc[i] / 1000, "{:.2f}")

# ---- stage table at NREF ----
cub, cubtc = g("cublas_sgemm"), g("cublas_tensorop_fp16")
rows = []
for k in order:
    gk = g(k)
    ref = cubtc if k in ("wmma_fp16_tc", "cublas_tensorop_fp16") else cub
    e = float(eps.loc[NREF, k]) if k in eps.columns else float("nan")
    rows.append((k, gk, 100 * gk / ref if ref else float("nan"), 100 * gk / P_FMA, e))
st = pd.DataFrame(rows, columns=["kernel", "gflops", "pct_ref", "pct_fma", "eps"])
with open("tab_ablation.tex", "w") as f:
    f.write("\\begin{tabular}{lrrrr}\n\\hline\nStage & GFLOP/s & \\% of cuBLAS$^\\dagger$ & \\% FMA peak & $\\varepsilon_F$ vs cuBLAS \\\\ \\hline\n")
    for r in st.itertuples():
        f.write("%s & %.0f & %.1f & %.1f & %.1e \\\\\n" % (tt(r.kernel), r.gflops, r.pct_ref, r.pct_fma, r.eps))
    f.write("\\hline\n\\end{tabular}")

setm("nRef", float(NREF))
setm("naiveG", naive_g); setm("uncoalG", g("naive_uncoalesced"))
setm("smemG", g("smem_tile32")); setm("regScalarG", g("reg2d_128x128x8_scalar"))
setm("regVecG", g("reg2d_128x128x8_vec")); setm("wmmaG", g("wmma_fp16_tc"))
setm("cublasG", cub); setm("cublasTCG", cubtc)
setm("regVecPctCublas", 100 * g("reg2d_128x128x8_vec") / cub, "{:.0f}")
setm("regVecPctFMA", 100 * g("reg2d_128x128x8_vec") / P_FMA, "{:.0f}")
setm("naivePctFMA", 100 * naive_g / P_FMA, "{:.1f}")
setm("cublasPctFMA", 100 * cub / P_FMA, "{:.0f}")
setm("wmmaPctTC", 100 * g("wmma_fp16_tc") / cubtc, "{:.0f}")
setm("vecGain", 100 * (g("reg2d_128x128x8_vec") / g("reg2d_128x128x8_scalar") - 1), "{:.0f}")
setm("coalGain", g("naive_m2") / g("naive_uncoalesced"), "{:.1f}")
setm("smemGain", g("smem_tile32") / naive_g, "{:.1f}")
setm("regGain", g("reg2d_128x128x8_scalar") / g("smem_tile32"), "{:.1f}")
setm("tcOverFP32", g("wmma_fp16_tc") / g("reg2d_128x128x8_vec"), "{:.1f}")

In [ ]:
# ============================================================
# ANALYSIS 2 -- tile-configuration sensitivity + model pruning (RQ2)
# ============================================================
tun = pd.read_csv("results_m3_tuning.csv")
NT = 4096 if 4096 in set(tun.N) else int(tun.N.max())
t = tun[tun.N == NT].copy().reset_index(drop=True)
t["ceiling"] = [ceiling(r.TM, r.TN, r.BM, r.BN) for r in t.itertuples()]
t = t.sort_values("gflops", ascending=False).reset_index(drop=True)

rho = spearmanr(t.ceiling, t.gflops).correlation if t.ceiling.nunique() > 1 else float("nan")
best = t.iloc[0]
DEFAULT = "r128x128x8_t8x8_v"
dflt = t[t.config == DEFAULT].iloc[0] if (t.config == DEFAULT).any() else best
top_ceiling = t.ceiling.max()
best_in_top = bool(best.ceiling >= 0.9 * top_ceiling)
pruned = t[t.ceiling < 0.5 * top_ceiling]
pruned_ok = bool((pruned.gflops < best.gflops * 0.9).all()) if len(pruned) else True

setm("tuneN", float(NT)); setm("tuneCount", float(len(t)))
setm("tuneBest", tt(best.config)); setm("tuneBestG", best.gflops)
setm("tuneDefaultG", dflt.gflops)
setm("tuneGain", 100 * (best.gflops / dflt.gflops - 1), "{:.0f}")
setm("tuneSpread", best.gflops / t.gflops.min(), "{:.1f}")
setm("tuneWorst", tt(t.iloc[-1].config)); setm("tuneWorstG", t.iloc[-1].gflops)
setm("modelRho", rho, "{:.2f}")
setm("modelPrunedFrac", 100 * len(pruned) / len(t), "{:.0f}")
setm("modelTopHit", "yes" if best_in_top else "no")
setm("modelPruneSafe", "yes" if pruned_ok else "no")
setm("bestRegs", float(best.regs)); setm("bestBlocks", float(best.blocks_per_sm))
print("best:", best.config, best.gflops, "| default:", dflt.config, dflt.gflops, "| rho:", rho)

fig, ax = plt.subplots(figsize=(3.5, 3.0))
y = np.arange(len(t))
ax.barh(y, t.gflops / 1000, color="tab:blue", label="measured")
ax.scatter(t.ceiling / 1000, y, marker="x", c="tab:red", s=14, label="model ceiling", zorder=3)
ax.set_yticks(y); ax.set_yticklabels(t.config, fontsize=5); ax.invert_yaxis()
ax.set_xlabel("TFLOP/s  (N=%d)" % NT); ax.legend(fontsize=6, loc="lower right")
savefig("fig_tuning")

sel = pd.concat([t.head(5), t.tail(3)])
with open("tab_tuning.tex", "w") as f:
    f.write("\\begin{tabular}{lrrrrr}\n\\hline\nConfig & Regs & Spill B & Blk/SM & GFLOP/s & Ceiling \\\\ \\hline\n")
    for r in sel.itertuples():
        f.write("%s & %d & %d & %d & %.0f & %.0f \\\\\n" % (tt(r.config), r.regs, r.spill_bytes, r.blocks_per_sm, r.gflops, r.ceiling))
    f.write("\\hline\n\\end{tabular}")

In [ ]:
# ============================================================
# ANALYSIS 3 -- numerical error budget of the FP16 Tensor-Core path (RQ3)
# ============================================================
pr = pd.read_csv("results_m3_precision.csv")
U16 = 2.0 ** -11
fig, ax = plt.subplots(figsize=(3.5, 2.5))
sty = {"signed": "-", "positive": "--"}
for dist in ["signed", "positive"]:
    d16 = pr[(pr.dist == dist) & (pr.kernel == "fp16_tensorcore")].sort_values("K")
    d32 = pr[(pr.dist == dist) & (pr.kernel == "fp32_reg2d")].sort_values("K")
    ax.loglog(d16.K, d16.eps_orig, sty[dist], marker="o", ms=2.5, c="tab:red", label="FP16-TC total (%s)" % dist)
    ax.loglog(d16.K, d16.eps_quant, sty[dist], marker="s", ms=2.5, c="tab:green", label="FP16-TC accum. only (%s)" % dist)
    ax.loglog(d32.K, d32.eps_orig, sty[dist], marker="^", ms=2.5, c="tab:blue", label="FP32 kernel (%s)" % dist)
ax.axhline(U16, c="k", ls=":", lw=0.8); ax.text(pr.K.min(), U16 * 1.15, "$u_{16}=2^{-11}$", fontsize=6)
ax.set_xlabel("K"); ax.set_ylabel(r"$\varepsilon_F$"); ax.legend(fontsize=5, ncol=1, loc="lower left")
savefig("fig_precision")

f16 = pr[pr.kernel == "fp16_tensorcore"]
fs, fp = f16[f16.dist == "signed"], f16[f16.dist == "positive"]
setm("epsSignedMin", fs.eps_orig.min(), "{:.1e}"); setm("epsSignedMax", fs.eps_orig.max(), "{:.1e}")
setm("epsPosMin", fp.eps_orig.min(), "{:.1e}"); setm("epsPosMax", fp.eps_orig.max(), "{:.1e}")
setm("epsAccMax", f16.eps_quant.max(), "{:.1e}")
setm("epsFPthirtytwoMax", pr[pr.kernel == "fp32_reg2d"].eps_orig.max(), "{:.1e}")
posK = fp.sort_values("K")
setm("posDecay", posK.eps_orig.iloc[0] / posK.eps_orig.iloc[-1], "{:.0f}")
setm("posKratio", posK.K.iloc[-1] / posK.K.iloc[0], "{:.0f}")
setm("quantOverAcc", (fs.eps_orig / fs.eps_quant).median(), "{:.0f}")

with open("tab_precision.tex", "w") as f:
    f.write("\\begin{tabular}{rrrrr}\n\\hline\n$K$ & \\multicolumn{2}{c}{signed inputs} & \\multicolumn{2}{c}{positive inputs} \\\\\n & total & accum. & total & accum. \\\\ \\hline\n")
    for K in sorted(f16.K.unique()):
        a = fs[fs.K == K].iloc[0]; b = fp[fp.K == K].iloc[0]
        f.write("%d & %.1e & %.1e & %.1e & %.1e \\\\\n" % (K, a.eps_orig, a.eps_quant, b.eps_orig, b.eps_quant))
    f.write("\\hline\n\\end{tabular}")

In [ ]:
# ============================================================
# ANALYSIS 4 -- panel-pipelined streaming GEMM (RQ4)
# ============================================================
pp = pd.read_csv("results_m3_pipeline.csv")
NP = 4096 if 4096 in set(pp.N) else int(pp.N.max())
q = pp[pp.N == NP]
comp = {r.variant: r.time_ms for r in q.itertuples() if r.variant.startswith("comp_")}
tA, tB, tK, tC = comp["comp_h2d_A"], comp["comp_h2d_B"], comp["comp_kernel"], comp["comp_d2h_C"]
page = float(q[q.variant == "pageable_serial"].time_ms.iloc[0])
pipe = q[q.variant == "pipelined"]
serial = float(pipe[(pipe.P == 1) & (pipe.S == 1)].time_ms.iloc[0])
bestrow = pipe.loc[pipe.time_ms.idxmin()]

def pipe_model(P):          # 3-stage pipeline, B pre-loaded, enough streams to keep engines busy
    a, k, c = tA / P, tK / P, tC / P
    return tB + a + k + c + (P - 1) * max(a, k, c)

fig, ax = plt.subplots(figsize=(3.5, 2.5))
for S in sorted(pipe.S.unique()):
    d = pipe[pipe.S == S].sort_values("P")
    ax.plot(d.P, d.time_ms, marker="o", ms=2.5, lw=1, label="measured, S=%d" % S)
Ps = np.array(sorted(pipe.P.unique()))
ax.plot(Ps, [pipe_model(p) for p in Ps], "k--", lw=1, label=r"model $\max(\cdot)$ (Eq. 6)")
ax.axhline(page, c="gray", ls=":", lw=1); ax.text(Ps.min(), page * 1.01, "pageable serial (M2-style)", fontsize=5.5)
ax.set_xscale("log", base=2); ax.set_xlabel("panels P"); ax.set_ylabel("end-to-end ms (N=%d)" % NP)
ax.legend(fontsize=5.5); savefig("fig_pipeline")

mdl_best = pipe_model(bestrow.P)
setm("pipeN", float(NP)); setm("pipeBestP", float(bestrow.P)); setm("pipeBestS", float(bestrow.S))
setm("pipeBestMs", bestrow.time_ms, "{:.1f}"); setm("pipeSerialMs", serial, "{:.1f}"); setm("pipePageMs", page, "{:.1f}")
setm("pipeSpeedup", serial / bestrow.time_ms, "{:.2f}"); setm("pipeSpeedupPage", page / bestrow.time_ms, "{:.2f}")
setm("pipeModelMs", mdl_best, "{:.1f}"); setm("pipeModelErr", 100 * (bestrow.time_ms - mdl_best) / mdl_best, "{:.0f}")
setm("pipeHideFrac", 100 * (serial - bestrow.time_ms) / (tA + tC + tK), "{:.0f}")
setm("tH2DA", tA, "{:.1f}"); setm("tH2DB", tB, "{:.1f}"); setm("tKern", tK, "{:.1f}"); setm("tD2H", tC, "{:.1f}")
dom = max([("H2D", tA), ("kernel", tK), ("D2H", tC)], key=lambda x: x[1])[0]
setm("pipeDominant", dom)

with open("tab_pipeline.tex", "w") as f:
    f.write("\\begin{tabular}{lrrrr}\n\\hline\n$N$ & Pageable & Pinned $P{=}1$ & Best pipelined ($P,S$) & Model \\\\ \\hline\n")
    for N in sorted(set(pp.N)):
        qq = pp[pp.N == N]; pi = qq[qq.variant == "pipelined"]
        cc = {r.variant: r.time_ms for r in qq.itertuples() if r.variant.startswith("comp_")}
        a_, b_, k_, c_ = cc["comp_h2d_A"], cc["comp_h2d_B"], cc["comp_kernel"], cc["comp_d2h_C"]
        br = pi.loc[pi.time_ms.idxmin()]
        m_ = b_ + a_ / br.P + k_ / br.P + c_ / br.P + (br.P - 1) * max(a_, k_, c_) / br.P
        s1 = float(pi[(pi.P == 1) & (pi.S == 1)].time_ms.iloc[0])
        pg = float(qq[qq.variant == "pageable_serial"].time_ms.iloc[0])
        f.write("%d & %.0f ms & %.0f ms & %.0f ms (%d,%d) & %.0f ms \\\\\n" % (N, pg, s1, br.time_ms, br.P, br.S, m_))
    f.write("\\hline\n\\end{tabular}")

In [ ]:
# ============================================================
# ANALYSIS 5 -- fused epilogue: the "why a custom kernel" case
# ============================================================
fu = pd.read_csv("results_m3_fusion.csv")
fp_ = fu.pivot(index="N", columns="variant", values="median_ms")
NF = 4096 if 4096 in fp_.index else int(fp_.index.max())
fp_["speedup_vs_cublas_pass"] = fp_["cublas_plus_epilogue_pass"] / fp_["custom_fused"]
fp_["extra_pass_ms_meas"] = fp_["cublas_plus_epilogue_pass"] - fp_["cublas_gemm_only"]
fp_["extra_pass_ms_model"] = [8.0 * N * N / (BW_DRAM * 1e6) for N in fp_.index]
fp_["fused_vs_unfused_custom"] = fp_["custom_plus_epilogue_pass"] / fp_["custom_fused"]
print(fp_.round(3).T)

fig, ax = plt.subplots(figsize=(3.5, 2.3))
xs = np.arange(len(fp_)); w = 0.27
ax.bar(xs - w, fp_["cublas_plus_epilogue_pass"], w, label="cuBLAS + epilogue pass")
ax.bar(xs, fp_["custom_plus_epilogue_pass"], w, label="custom + epilogue pass")
ax.bar(xs + w, fp_["custom_fused"], w, label="custom fused")
ax.set_xticks(xs); ax.set_xticklabels(fp_.index); ax.set_xlabel("N"); ax.set_ylabel("ms"); ax.set_yscale("log")
ax.legend(fontsize=5.5); savefig("fig_fusion")

r = fp_.loc[NF]
setm("fuseN", float(NF)); setm("fuseSpeedup", r["speedup_vs_cublas_pass"], "{:.2f}")
setm("fuseVsCustom", r["fused_vs_unfused_custom"], "{:.2f}")
setm("fuseExtraMeas", r["extra_pass_ms_meas"], "{:.2f}"); setm("fuseExtraModel", r["extra_pass_ms_model"], "{:.2f}")
setm("fuseCublasMs", r["cublas_plus_epilogue_pass"], "{:.1f}"); setm("fuseFusedMs", r["custom_fused"], "{:.1f}")
setm("fuseCublasOnlyMs", r["cublas_gemm_only"], "{:.1f}")
setm("fuseWinN", ", ".join(str(int(n)) for n in fp_.index[fp_["speedup_vs_cublas_pass"] > 1.0]) or "none")
setm("fuseEps", float(fu[fu.N == NF].eps_vs_cublas_plus_pass.iloc[0]), "{:.1e}")

with open("tab_fusion.tex", "w") as f:
    f.write("\\begin{tabular}{rrrrr}\n\\hline\n$N$ & cuBLAS+pass & custom fused & speedup & extra pass (meas./model) \\\\ \\hline\n")
    for N, rr in fp_.iterrows():
        f.write("%d & %.2f ms & %.2f ms & %.2f$\\times$ & %.2f / %.2f ms \\\\\n" % (N, rr["cublas_plus_epilogue_pass"], rr["custom_fused"], rr["speedup_vs_cublas_pass"], rr["extra_pass_ms_meas"], rr["extra_pass_ms_model"]))
    f.write("\\hline\n\\end{tabular}")

In [ ]:
# ============================================================
# ANALYSIS 6 -- comparison with Milestone 2 + write all LaTeX artefacts
# ============================================================
rows = []
m2 = pd.read_csv("results_milestone2.csv") if os.path.exists("results_milestone2.csv") else None
for N in [256, 512, 1024, 2048, 4096]:
    if N not in piv.index: continue
    m2g = float(m2[m2.N == N].cuda_naive_gflops.iloc[0]) if (m2 is not None and (m2.N == N).any()) else float("nan")
    rows.append((N, m2g, g("naive_m2", N), g("reg2d_128x128x8_vec", N), g("wmma_fp16_tc", N), g("cublas_sgemm", N)))
cmpdf = pd.DataFrame(rows, columns=["N", "m2", "m3naive", "regvec", "wmma", "cublas"])
with open("tab_m2cmp.tex", "w") as f:
    f.write("\\begin{tabular}{rrrrrr}\n\\hline\n$N$ & M2 naive & M3 naive & M3 reg-blk & M3 WMMA & cuBLAS \\\\ \\hline\n")
    for r in cmpdf.itertuples():
        f.write("%d & %.0f & %.0f & %.0f & %.0f & %.0f \\\\\n" % (r.N, r.m2, r.m3naive, r.regvec, r.wmma, r.cublas))
    f.write("\\hline\n\\end{tabular}")
r4 = cmpdf[cmpdf.N == NREF].iloc[0] if (cmpdf.N == NREF).any() else None
if r4 is not None:
    setm("m2NaiveG", r4.m2); setm("m3OverM2Reg", r4.regvec / r4.m2, "{:.1f}"); setm("m3OverM2TC", r4.wmma / r4.m2, "{:.1f}")
    setm("m3OverM2Cublas", r4.cublas / r4.m2, "{:.1f}")
    setm("m2RemeasureDelta", 100 * (r4.m3naive / r4.m2 - 1), "{:+.0f}")
small = cmpdf[cmpdf.N == 256]
if len(small):
    setm("m2SmallDelta", 100 * (float(small.m3naive.iloc[0]) / float(small.m2.iloc[0]) - 1), "{:+.0f}")

# --- validation summary ---
if os.path.exists("results_m3_validation.csv"):
    va = pd.read_csv("results_m3_validation.csv")
    setm("valRows", float(len(va))); setm("valFail", float((va.status != "PASS").sum()))
    setm("valShapes", float(va[["M", "N", "K"]].drop_duplicates().shape[0]))
    setm("valWorstFP32", va[va.kernel != "wmma_fp16_tc"].eps_vs_ref.max(), "{:.1e}")
    setm("valWorstQuant", va[va.kernel == "wmma_fp16_tc"].eps_vs_quantref.max(), "{:.1e}")

# --- write macros + show summary ---
with open("results_macros.tex", "w") as f:
    f.write("% auto-generated by Milestone_3.ipynb -- do not edit\n")
    for k, v in sorted(MAC.items()):
        f.write("\\def\\%s{%s}\n" % (k, v))
print("macros written:", len(MAC))
for k, v in sorted(MAC.items()): print("  %-18s %s" % (k, v))

In [ ]:
# ============================================================
# Bundle everything the LaTeX report needs
# ============================================================
files = (glob.glob("results_m3_*.csv") + glob.glob("fig_*.pdf") + glob.glob("fig_*.png") +
         glob.glob("tab_*.tex") + ["results_macros.tex", "gemm_cuda_opt.cu"])
with zipfile.ZipFile("m3_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in files:
        if os.path.exists(fn): z.write(fn)
print("bundled %d files into m3_artifacts.zip" % len(files))
print("-> upload the contents of this zip next to Milestone_3.tex (e.g. in Overleaf) and compile.")
try:
    from google.colab import files as colab_files
    colab_files.download("m3_artifacts.zip")
except Exception as e:
    print("(not on Colab, download m3_artifacts.zip manually)")